# Deploy model as a webservice on Azure Kubernetes Service

## Table of contents
1. [Prerequisites](#prerequisites)

2. [Initialize workspace](#workspace)

3. [Deploy Model to AKS](#deploymodel)

- a) [Create scoring file](#scoringfile)
- b) [Define Enviroment](#env)
- c) [Deployment configuration](#configfile)
- d  [Deploy Webservice](#webservice)
- e) [Test Webservice](#testservice)



### 1. Prerequisites <a id='prerequisites'></a>

In [12]:
import numpy as np
import azureml.core

# display the core SDK version number
print("Azure ML SDK Version: ", azureml.core.VERSION)

Azure ML SDK Version:  1.61.0


### 2. Initialize workspace <a id='workspace'></a>

In [13]:
from azureml.core import Workspace
from azureml.core.model import Model

subscription_id = 'f2f70602-65d9-479b-8014-a1c92a06ef0e'
resource_group = 'Learn_MLOps'
workspace_name = 'MLOps'

ws = Workspace(subscription_id, resource_group, workspace_name)
print(ws.name, ws.resource_group, ws.location, sep='\n')

MLOps
Learn_MLOps
eastus2


### 3. Deploy model <a id='deploymodel'></a>

#### a) Create a scoring script <a id='scoringfile'></a>

In [ ]:
%%writefile score.py
import numpy as np
import os
import glob
import joblib
import onnxruntime
from inference_schema.schema_decorators import input_schema, output_schema
from inference_schema.parameter_types.numpy_parameter_type import NumpyParameterType

def init():
    global model, scaler, input_name, label_name

    scaler_path = glob.glob(
        os.path.join(os.getenv('AZUREML_MODEL_DIR'), 'scaler', '**', '*.pkl'),
        recursive=True
    )[0]
    scaler = joblib.load(scaler_path)

    model_onnx = glob.glob(
        os.path.join(os.getenv('AZUREML_MODEL_DIR'), 'support-vector-classifier', '**', '*.onnx'),
        recursive=True
    )[0]
    model = onnxruntime.InferenceSession(model_onnx, None)
    input_name = model.get_inputs()[0].name
    label_name = model.get_outputs()[0].name


@input_schema('data', NumpyParameterType(np.array([[34.927778, 0.24, 7.3899, 83, 16.1000, 1016.51, 1]])))
@output_schema(NumpyParameterType(np.array([0])))
def run(data):
    try:
        data = scaler.transform(data.reshape(1, 7))
        result = model.run([label_name], {input_name: data.astype(np.float32)})[0]
    except Exception as e:
        return str(e)
    return result.tolist()

#### b) Define Environment <a id='env'></a>

In [19]:
from azureml.core.environment import Environment
from azureml.core.conda_dependencies import CondaDependencies
from azureml.core.model import InferenceConfig

# Fresh Python 3.8 env — AzureML-Minimal uses py3.6, incompatible with scikit-learn>=1.0
env = Environment(name="weather-deploy-env")
conda_deps = CondaDependencies()
conda_deps.set_python_version("3.8")
env.python.conda_dependencies = conda_deps

In [20]:
for pip_package in ["numpy", "onnxruntime", "joblib", "scikit-learn",
                    "inference-schema[numpy-support]", "azureml-defaults"]:
    env.python.conda_dependencies.add_pip_package(pip_package)

inference_config = InferenceConfig(entry_script='score.py', environment=env)

#### c) Deployment Configuration <a id='configfile'></a>

In [21]:
from azureml.core.webservice import AksWebservice

aks_config = AksWebservice.deploy_configuration(collect_model_data=True)

#### d) Deploy web service <a id='webservice'></a>

In [2]:
from azureml.core.compute import ComputeTarget, AksCompute
from azureml.core.compute_target import ComputeTargetException

aks_name = 'port-aks'

try:
    aks_target = ComputeTarget(workspace=ws, name=aks_name)
    if aks_target.provisioning_state == 'Failed':
        print(f'Cluster "{aks_name}" is in Failed state — deleting and recreating...')
        aks_target.delete()
        aks_target.wait_for_completion(show_output=True, is_delete_operation=True)
        raise ComputeTargetException('Deleted failed cluster, recreating.')
    print(f'Found existing cluster "{aks_name}" in state: {aks_target.provisioning_state}')
except ComputeTargetException:
    # Standard_D3_v2 is not available in eastus2 — use Standard_D2s_v3 instead
    prov_config = AksCompute.provisioning_configuration(vm_size='Standard_D2s_v3')
    aks_target = ComputeTarget.create(workspace=ws, name=aks_name,
                                      provisioning_configuration=prov_config)

if aks_target.get_status() != 'Succeeded':
    aks_target.wait_for_completion(show_output=True)

print('AKS cluster state:', aks_target.get_status())

NameError: name 'ws' is not defined

In [23]:
model1 = Model(ws, 'scaler')
model2 = Model(ws, 'support-vector-classifier')

service_name = 'weather-aks-prediction'

In [26]:
service = Model.deploy(ws, service_name, models=[model1, model2], inference_config=inference_config, deployment_config=aks_config, deployment_target=aks_target,overwrite=True)
service.wait_for_deployment(show_output = True)
print(service.state)

/tmp/ipykernel_68888/2583786652.py:1: FutureWarning: azureml.core.model:
To leverage new model deployment capabilities, AzureML recommends using CLI/SDK v2 to deploy models as online endpoint, 
please refer to respective documentations 
https://docs.microsoft.com/azure/machine-learning/how-to-deploy-managed-online-endpoints /
https://docs.microsoft.com/azure/machine-learning/how-to-attach-kubernetes-anywhere 
For more information on migration, see https://aka.ms/acimoemigration 
To disable CLI/SDK v1 deprecation warning set AZUREML_LOG_DEPRECATION_WARNING_ENABLED to 'False'
  service = Model.deploy(ws, service_name, models=[model1, model2], inference_config=inference_config, deployment_config=aks_config, deployment_target=aks_target,overwrite=True)


Tips: You can try get_logs(): https://aka.ms/debugimage#dockerlog or local deployment: https://aka.ms/debugimage#debug-locally to debug if deployment takes longer than 10 minutes.
Running
2026-05-06 17:14:38-07:00 Creating Container Registry if not exists.
2026-05-06 17:14:38-07:00 Use the existing image.
2026-05-06 17:14:45-07:00 Checking the status of deployment weather-aks-prediction..
2026-05-06 17:14:58-07:00 Checking the status of inference endpoint weather-aks-prediction.
Succeeded
AKS service creation operation finished, operation "Succeeded"
Healthy


In [27]:
print(service.get_logs())

/bin/bash: /azureml-envs/azureml_aae9930899053d29a1d162c049d1285e/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /azureml-envs/azureml_aae9930899053d29a1d162c049d1285e/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /azureml-envs/azureml_aae9930899053d29a1d162c049d1285e/lib/libtinfo.so.6: no version information available (required by /bin/bash)
2026-05-07T00:14:50,203049774+00:00 - rsyslog/run 
bash: /azureml-envs/azureml_aae9930899053d29a1d162c049d1285e/lib/libtinfo.so.6: no version information available (required by bash)
2026-05-07T00:14:50,210411533+00:00 - gunicorn/run 
2026-05-07T00:14:50,213888160+00:00 | gunicorn/run | 
2026-05-07T00:14:50,216201079+00:00 | gunicorn/run | ###############################################
2026-05-07T00:14:50,218308596+00:00 | gunicorn/run | AzureML Container Runtime Information
2026-05-07T00:14:50,220417312+00:00 | gunicorn/run | ########################################

In [28]:
service.update(enable_app_insights=True)

#### e) Test web service <a id='testservice'></a>

In [29]:
print(service.scoring_uri)

http://20.7.107.211:80/api/v1/service/weather-aks-prediction/score


In [30]:
print(service.swagger_uri)

http://20.7.107.211:80/api/v1/service/weather-aks-prediction/swagger.json


In [31]:
service.state

'Healthy'

In [32]:
import json


input_payload = json.dumps({
    'data': [[34.927778, 0.24, 7.3899, 83, 16.1000, 1016.51, 1]],
    'method': 'predict'  # If you have a classification model, you can get probabilities by changing this to 'predict_proba'.
})

output = service.run(input_payload)

print(output)

[1]


In [ ]:
# service.delete()